In [4]:
import os 
import json
from minsearch import Index
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv() # get api key 

openai_client = OpenAI(
    api_key=os.getenv("CEREBRAS_API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

from ingest_hm import load_faq_data, build_index
from rag_helper_hm import RAGBase

In [5]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
"""

# user prompt has placeholders for the question and context 
USER_PROMPT_TEMPLATE = """

Question: 
{question}

Context: 
{context}

"""

In [6]:
documents = load_faq_data()
index = build_index(documents)

In [7]:
# Q1: How many lesson pages are in the dataset? 
print(f"Q1: We have {len(documents)} lesson pages in our dataset")

Q1: We have 72 lesson pages in our dataset


In [8]:
# search 
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query, num_results=5) 

# Q2: What's the filename of the first result?
print("Q2: The filename of the first result is: ", search_results[0])

Q2: The filename of the first result is:  {'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, 

In [9]:
# building the context; parsing 
def build_context(search_results): 

    lines =[]
    for doc in search_results: 
        lines.append(doc["content"])
        lines.append(doc["filename"])
        lines.append("")

    return  "\n".join(lines).strip()

def build_prompt(query, search_results):
    
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = query, 
        context = context
    )
    return prompt.strip()

In [10]:
prompt = build_prompt(query, search_results)
print(prompt)

Question: 
How does the agentic loop keep calling the model until it stops?

Context: 
# The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Too

In [11]:
# responses API (OpenAI)
input=[
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model="zai-glm-4.7",
    messages=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]
)

In [12]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Based on the provided context, the agentic loop keeps calling the model using a `while` loop that manages the flow of messages and tool execution.

Here is how it works:

1.  **The Loop Structure:** The code uses a `while True` loop to repeatedly call the model API with the current message history.
2.  **Processing the Response:** Inside the loop, the code iterates through the model's output. If it finds an item of type `function_call`, it executes that function, formats the result, and appends the result to the message history. It also sets a boolean flag (e.g., `has_function_calls`) to `True`.
3.  **The Exit Condition:** After processing the output, the loop checks the `has_function_calls` flag. If the model **did not** make any function calls in the current turn (meaning it provided a final text message), the flag remains `False`.
4.  **Stopping:** The loop includes a condition `if has_function_calls == False: break`. This ensures the loop exits only when the model stops asking for 

In [13]:
print(usage)

7182


In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Q4: We get {len(chunks)} chunks")

Q4: We get 295 chunks


In [15]:
chunked_documents = chunk_documents(documents)   
index_c = build_index(chunked_documents)     # using chunks instead        

assistant = RAGBase(
    index=index_c,
    llm_client=openai_client,
)

In [16]:
answer_c, usage_c = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Based on the provided context, the agentic loop keeps calling the model using a `while` loop that manages the flow of messages and tool execution.

Here is how it works:

1.  **The Loop Structure:** The code uses a `while True` loop to repeatedly call the model API with the current message history.
2.  **Processing the Response:** Inside the loop, the code iterates through the model's output. If it finds an item of type `function_call`, it executes that function, formats the result, and appends the result to the message history. It also sets a boolean flag (e.g., `has_function_calls`) to `True`.
3.  **The Exit Condition:** After processing the output, the loop checks the `has_function_calls` flag. If the model **did not** make any function calls in the current turn (meaning it provided a final text message), the flag remains `False`.
4.  **Stopping:** The loop includes a condition `if has_function_calls == False: break`. This ensures the loop exits only when the model stops asking for 

In [17]:
print(usage_c)

2308


In [18]:
print(f"The usage is now {usage/usage_c:.2f} times lower")

The usage is now 3.11 times lower


In [ ]:
# Agentic RAG
def search(query: str, num_results: int = 5) -> list:
    """Search the course FAQ index for relevant documents."""
    return index_c.search(query, num_results=num_results)

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course FAQ index for relevant documents.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "num_results": {"type": "integer", "default": 5}
            },
            "required": ["query"]
        }
    }
}

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [ ]:
def agent_loop(instructions, question, model="zai-glm-4.7") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    search_count = 0  

    for i in range(10):
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                search_count += 1  # count here
                print(f"search #{search_count}:", tc.function.arguments)
                result = search(**json.loads(tc.function.arguments))
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, indent=2)
                })
        else:
            print(f"Total searches: {search_count}")
            return msg.content

    return msg.content


In [20]:
agent_loop(instructions, "How does the agentic loop work, and how is it different from plain RAG?")

search #1: {"query": "agentic loop", "num_results": 5}
search #2: {"query": "RAG retrieval augmented generation", "num_results": 5}
search #3: {"query": "agentic loop vs RAG difference", "num_results": 5}
search #4: {"query": "\"agentic loop\" while loop function calling", "num_results": 5}
search #5: {"query": "agents intro agentic loop by hand", "num_results": 5}
Total searches: 5


'Based on the course materials, here\'s a clear explanation of the agentic loop and how it differs from plain RAG:\n\n## The Agentic Loop\n\nThe agentic loop is an agent pattern where the **LLM is in the driver\'s seat**. It consists of a `while` loop that:\n\n1. **Calls the LLM** with the current message history\n2. **Executes any tool/function calls** the LLM returns\n3. **Sends the tool results back** to the LLM\n4. **Repeats** until the model produces a final answer with no more tool calls\n\n### Core components:\n- **Instructions**: A developer message defining the agent\'s role and behavior\n- **Tools**: Functions the agent can call (e.g., search, web browsing)\n- **Memory**: Message history tracking every prompt, model output, and tool result\n\n```python\nwhile True:\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n    \n    # Execute any function calls and append results\n    for item 